# WIT-metrics

Jupyter notebook

Calculate inundation summary metrics from Geoscience Australia Wetland Insights Tool (WIT) data (csv files)

[https://knowledge.dea.ga.gov.au/data/product/dea-wetlands-insight-tool-ramsar-wetlands/](https://knowledge.dea.ga.gov.au/data/product/dea-wetlands-insight-tool-ramsar-wetlands/)

[Dunn, B., Ai, E., Alger, M.J. et al. Wetlands Insight Tool: Characterising the Surface Water and Vegetation Cover Dynamics of Individual Wetlands Using Multidecadal Landsat Satellite Data. Wetlands 43, 37 (2023). https://doi.org/10.1007/s13157-023-01682-7](https://link.springer.com/article/10.1007/s13157-023-01682-7)

## Lineage

This code was derived from some initial code Geoscience Australia contributed to an MDBA project "BWS Vulnerabilities"
Changes since include:

* modernised code for Pandas 2.0+, Python 3.11 
* batch input of multiple WIT CVS in a folder (currently the ANAEv3 WIT output includes 270,653 polygons, each with its own csv file)
* multiple processor pool support to speed execution when running on a PC workstation
* linear interpolation of the observations dates to daily data to improve estimates of inundation duration, the interpolated data permits estimation of monthly stats
* some bug fixes in the inundation event metrics that were required when using the interpolated data
* output formatting

## Dependencies

* A folder containing WIT csv (obtained for the BWS Priorities Project from Geoscience Australia for each ANAE polygon > 1Ha)
* The `wit_metrics_worker.py` module in the same folder as the notebook contains the program code.  This notebook is just the orchestrator to configure and run it.

## Optional

* A shapefile (or equivalent) that contains the area that the WIT result was run over.
  
     
## Background

The WIT data are generated by DEA with given wetland polygons and stored in a database on NCI. The data can be dumped into a csv when required. This notebook provides a way in computing temporal statistics (metrics) from the WIT csv.

## WIT Data definition

* WIT csv data files provide the following metrics for each polygon unit

      feature_id: The unique identifier for the polygon
      date: time of observation
      bs: percentage of bare soil
      npv: percentage of non photosynthetic vegetation
      pv: percentage of green/photosynthetic vegetation
      wet: percentage of wetness
      water: percentage of water
      pc_missing: the proportion of missing pixels in the polygon (cloud cover, satellite sensor issues)

## Description

This notebook uses existing WIT data to compute metrics.

* First we load the existing WIT csv data from a saved csv location
* Then we compute the metrics for all polygons and output the results to CSV files.  The input CSV files are processed in "batches" that are spread across multiple CPU cores.  When execution is complete the various batch outputs are merged together into single result files that contain metrics for every CSV feature ID (e.g. ANAE polygons)

The following files are created:

* **RESULT_WIT_yearly_metrics**: min, max, mean, median of each WIT metric per calendar year
* **RESULT_WIT_event_threshold**: Uses an adaptive approach (30th percentile capped by floor of 5% area of polygon to prevent trivial inundation and noise being an inundation event in very dry sites, and ceiling of 50% so that very wet sites are still considered inundated until they dry to < 50% area).
* **RESULT_WIT_time_since_last_inundation**: number of days since the inundation event threshold was exceeded
* **RESULT_WIT_inundation_metrics**: this is a join of the RESULT_WIT_ANAE_event_time and RESULT_WIT_ANAE_event_stats

If the optional debug_event_stats is set True then two intermediate files that are joined to make "RESULT_WIT_ANAE_inundation_metrics" are also saved:

* **RESULT_WIT_event_times**: start and end time, duration, duration of preceding gap (the dry period)
* **RESULT_WIT_event_stats**: for each event calculates the area of the polygon that was wet using the combination water+wet

### Processing Environment

Python 3.11.11

install requirements

```pip install -r requirements.txt```

### Execution

load this ```wit-metrics.ipynb``` into a Jupyter iPython environment and step through the notebook.  It will export the configuration entries to a python file ```config.py``` and will execute ```wit_metrics_worker.py``` that runs outside of jupyter to speed up the code with parallel processing across available CPU cores

The `wit_metrics_worker.py` can also be run without the notebook and integrated into other workflows. It reads the configuration from `config.py`

### Alternative method of execution in pure Python

the wit_metrics_worker.py module that is exported from the notebook (a copy is included in repository) can be run by itself.  First edit the config.py and then run wit_metrics_worker.py from the command line. 

```python wit_metrics_worker.py```
    
## Contact

Dr Shane Brooks
https://brooks.eco

![Brooks.eco logo](brooks-logo.png "Brooks Ecology & Technology")


# User Defined Parameters
are written to config.py to be read into python scripts 

In [ ]:
%%writefile config.py

""" -----------------------------------------------------------------
    Config parameters are written to a python module config.py which is
    imported by the main processing worker script wit_metrics_worker.py
    #
    This file is generated by the wit-metrics.ipynb notebook and should not be modified directly.
    Instead, modify the parameters in the notebook and then run the notebook to generate the config.py file.
    -----------------------------------------------------------------
"""
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict
import os
import multiprocessing as mp


@dataclass(frozen=True)
class WITMetricsConfig():
    BASE_DIR: Path = Path(__file__).parent
    INPUT_DIR: Path = BASE_DIR / "input"
    OUTPUT_DIR: Path = BASE_DIR / "output"
    LOG_DIR: Path = BASE_DIR / "log"

    # shapefile: the shape file mentioned above to find the  and get their area
    # set to '' to disable area lookup
    SHAPEFILE_PATH: Path = INPUT_DIR / "shp/ANAEv3_test.shp"

    # shapefile field name that identifies each polygon -  the ANAEv3 UID geohash was used here.
    # The ANAE UID is also used in the naming convention for the CSV files
    SHAPEFILE_KEY: str = "UID"

    # Path to folder that contains the WIT csv files to process.  When debugging providing a single file might be prudent.
    WIT_CSV_PATH: Path = INPUT_DIR / "csv"

    # Only use WIT data where the pc_missing is less than the threshold (default is 0.1) i.e. >90% of the polygon was visible to satellites
    PC_MISSING_THRESHOLD: float = 0.1

    # csv feature_id - the WIT csv output files include a column 'feature_id' that in this case is the ANAE UID
    WIT_FEATURE_ID: str = "feature_id"

    # whether to interpolate the WIT observation dates (typically 10-50+ per year) to daily data (365 per year)
    # This is computationally expensive but improves estimates of inundation duration and time since last inundation.
    # Monthly WIT stats require interpolated daily data to infill missing records and will not be generated if interpolate_to_daily = False
    INTERPOLATE_TO_DAILY: bool = True

    # set to True to save the interpolated daily WIT csv in a subfolder under the csv_files (unnecessary and take up a lot of space but good for debugging)
    SAVE_INTERPOLATED_CSV: bool = False

    # monthly metrics files are too big for most computers when all metrics are used (e.g. just 4 metrics x 270,000 polygons x 450 months is a 5GB csv file)
    # specify a subset that will be joined together into the monthly result - must include ["feature_id","date", at-least-one-metric]\
    MONTHLY_SUBSET: list | None = None  # do not prune
    # monthly_subset=[
    #         "feature_id",
    #         "date",
    #         "water_median",
    #         "wet_median",
    #         "pv_median",
    #         "npv_median",
    #         "bs_median",
    #         "count",
    #     ]

    # set to true to save intermediate data frames containing the event times and stats
    # these are saved in the working directory
    DEBUG_EVENT_TIMES: bool = False

    # batchsize is the number of WIT csv files to include in each 'batch' that is processed by each single CPU core.
    # The code was designed to process several 100,000 polygons in small batches that fit into the computer memory
    # then glue all the batch results together at the end.
    # On a workstation with 16 cpu cores and 64MB RAM a batchsize of 100-200 worked well. With smaller number of CSV a batch size < total number of CSV allows the
    # calculations to be spread across multiple processors.
    # during processing the code will generate outputs for each batch then glue them together at the end.

    BATCH_SIZE: int = 100

    # tag prepended to final result files (zipped csv)
    TAG: str = "RESULT"

    # Whether to zip the final result csv to save space (python/pandas can read the csv from the zips)
    ZIP_RESULT: bool = True

# ###########################################################

wit_metrics_cfg = WITMetricsConfig()



Overwriting config.py


# write wit_metrics_worker.py python module to the specified working directory
* To speed up the processing of many CSV files (we initially processed 270,653 ANAE csv) we divide the work across multiple processor cores.  To achieve this the notebook calls an external worker module contains the routines for summarising the WIT CSV data


# Load packages
Import Python packages that are used for the analysis.

Use standard import commands; some are shown below. 


In [30]:
import os
import sys
import glob

from tqdm.notebook import tqdm
import time
import multiprocessing


#uncomment if running in ArcGIS Pro environment and GDAL errors are encountered
# os.environ["GDAL_DATA"] = "C:/Program Files/ArcGIS/Pro/Resources/pedata/gdaldata"

sys.path.append(
    os.getcwd()
)  # appends cwd to path allowing python to find config.py and wit_metrics_worker.py in the notebook directory
print (os.getcwd())




d:\wit-metrics


# multiprocessing code

this is the loop that executes to compute the metrics for all CSV in the specified path using multithreaded workers


In [31]:
#-----------------------------------------------------------------
# Reload the config and wit_metrics_worker in case they were edited
#-----------------------------------------------------------------
try:
    del sys.modules['config']
    del sys.modules['wit_metrics_worker']
except:
    pass

from config import wit_metrics_cfg as config
import wit_metrics_worker as worker


#-----------------------------------------------------------------
# code is under if __name__ == "__main__": to permit multiprocessing in jupyter notebooks
#-----------------------------------------------------------------
if __name__ == "__main__":

    os.makedirs(config.OUTPUT_DIR, exist_ok=True)

    output_filenames = [
        "WIT_yearly_metrics",
        "WIT_event_threshold",
        "WIT_inundation_metrics",
        "WIT_time_since_last_inundation",
        "WIT_event_times",
        "WIT_event_stats",
    ]

    if config.INTERPOLATE_TO_DAILY:
        output_filenames.append("WIT_monthly_metrics")

    # Cleanup old batch outputs
    worker.delete_old_batch_outputs(config.OUTPUT_DIR, output_filenames)

    csv_list = glob.glob(os.path.join(config.WIT_CSV_PATH, "*.csv"))
    total_files = len(csv_list)

    if total_files == 0:
        raise RuntimeError(f"No CSV files found in {config.WIT_CSV_PATH}")
    
    print(f"Found {total_files} WIT CSV files in folder {config.WIT_CSV_PATH}")

Found 6 WIT CSV files in folder D:\wit-metrics\input\csv


## Setup multiprocessing environment

In [32]:

# Conservative core usage (important for IO-bound workloads)
CPU = min(total_files, multiprocessing.cpu_count() // 2)

batch_size = config.BATCH_SIZE
chunk_size = batch_size * CPU

print(f"Processing {total_files} WIT CSV using {CPU} worker processes")
print(f"Maximum batch size per process: {batch_size}")

    

Processing 6 WIT CSV using 6 worker processes
Maximum batch size per process: 100


In [33]:
start = time.process_time()

# ------------------------------------------------------------------
# Batch-level multiprocessing only
# ------------------------------------------------------------------
for j in tqdm(range(0, total_files, chunk_size), desc="Processing chunks"):
    mpbatch = csv_list[j : j + chunk_size]

    work = []
    for i in range(0, len(mpbatch), batch_size):
        batch_files = mpbatch[i : i + batch_size]
        chunk_id = j + i
        work.append((batch_files, chunk_id))

    with multiprocessing.Pool(processes=min(len(work), CPU)) as pool:
        pool.starmap(worker.process_batch, work)

print(
    f"{total_files} CSVs processed in "
    f"{time.strftime('%H:%M:%S', time.gmtime(time.process_time() - start))}"
)


Processing chunks:   0%|          | 0/1 [00:00<?, ?it/s]

6 CSVs processed in 00:00:00


In [34]:

# ------------------------------------------------------------------
# Merge batch outputs
# ------------------------------------------------------------------
worker.merge_batches(
    config.OUTPUT_DIR,
    output_filenames,
    monthly_subset=config.MONTHLY_SUBSET,
    tag=config.TAG
)

print("All batches merged successfully.")

   

Merging 1 batch outputs into RESULT_WIT_yearly_metrics.csv ...


100%|██████████| 1/1 [00:00<00:00, 118.60it/s]


Merging 1 batch outputs into RESULT_WIT_event_threshold.csv ...


100%|██████████| 1/1 [00:00<00:00, 426.90it/s]


Merging 1 batch outputs into RESULT_WIT_inundation_metrics.csv ...


...

Merging 1 batch outputs into RESULT_WIT_time_since_last_inundation.csv ...


100%|██████████| 1/1 [00:00<00:00, 287.68it/s]


Merging 1 batch outputs into RESULT_WIT_monthly_metrics.csv ...


100%|██████████| 1/1 [00:00<00:00, 101.95it/s]

All batches merged successfully.


In [35]:
#check the outputs look ok and that batches merged properly into RESULT_WIT_* files

# Optional cleanup
worker.delete_old_batch_outputs(config.OUTPUT_DIR, output_filenames)

### END